# Lesson 9A: Convolutional Neural Network Theory

<a name="introduction"></a>
## Introduction

Lesson 3 covered fully-connected neural networks: every unit connects to every
unit in the adjacent layer. For image data, this is wasteful and statistically
naive — a 200x200 RGB image has 120,000 input values, so a single
fully-connected hidden layer of modest size already needs tens of millions of
parameters, with no built-in notion that a pattern (an edge, a texture) means
the same thing wherever it appears in the image.

Convolutional Neural Networks (CNNs) encode two structural assumptions
directly into the architecture:

1. **Local connectivity**: a unit responds to a small spatial neighborhood of
   the input, not the entire image at once
2. **Weight sharing**: the same small set of weights (a filter) is applied at
   every spatial location, so a pattern learned in one part of the image is
   recognized anywhere else it appears

Both assumptions are *inductive biases* — they are not learned, they are built
into the architecture, and they are exactly the right bias for image data
(unlike, say, tabular data with no spatial structure).

In this lesson, we'll:
1. Derive discrete convolution and the output size formula (padding, stride)
2. Derive weight sharing's parameter-reduction and the growth of receptive field with depth
3. Derive backpropagation through convolutional and pooling layers from first principles
4. Implement Conv2D and MaxPool2D from scratch in NumPy, with both forward and backward passes verified by numerical gradient checking
5. Train a small from-scratch CNN on real MNIST digits and analyze the training dynamics
6. Briefly motivate why very deep CNNs need skip connections (ResNets)

Lesson 9b applies these ideas with PyTorch, transfer learning, and feature visualization.


## Table of Contents

1. [Introduction](#introduction)
2. [Required Libraries](#required-libraries)
3. [Discrete Convolution](#discrete-convolution)
   - [1D Intuition](#1d-intuition)
   - [2D Discrete Convolution](#2d-discrete-convolution)
   - [Output Size: Padding and Stride](#output-size-padding-and-stride)
4. [Weight Sharing and Parameter Reduction](#weight-sharing-and-parameter-reduction)
5. [Receptive Field](#receptive-field)
6. [Backpropagation Through Convolutional Layers](#backpropagation-through-convolutional-layers)
   - [Gradient with Respect to the Filter](#gradient-with-respect-to-the-filter)
   - [Gradient with Respect to the Input](#gradient-with-respect-to-the-input)
7. [Pooling Layers](#pooling-layers)
   - [Max Pooling and Its Gradient](#max-pooling-and-its-gradient)
   - [Average Pooling and Its Gradient](#average-pooling-and-its-gradient)
8. [From-Scratch Implementation](#from-scratch-implementation)
   - [Conv2D Layer](#conv2d-layer)
   - [MaxPool2D Layer](#maxpool2d-layer)
   - [Gradient Checking](#gradient-checking)
9. [Training a From-Scratch CNN on MNIST](#training-a-from-scratch-cnn-on-mnist)
   - [Network Architecture](#network-architecture)
   - [Training Loop and Convergence](#training-loop-and-convergence)
10. [Why Very Deep CNNs Need Skip Connections](#why-very-deep-cnns-need-skip-connections)
11. [Conclusion](#conclusion)
    - [Key Insights](#key-insights-7)
    - [Further Reading](#further-reading-7)


<a name="required-libraries"></a>
## Required Libraries

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
np.random.seed(42)


<a name="discrete-convolution"></a>
## Discrete Convolution

<a name="1d-intuition"></a>
### 1D Intuition

Discrete convolution slides a small filter (kernel) over an input signal,
computing a weighted sum at every position:

$$y[i] = \sum_{k} w[k] \, x[i + k]$$

(Strictly, mathematical convolution flips the kernel: $y[i] = \sum_k w[k] x[i-k]$.
Every deep learning framework, including the implementation below, actually
computes *cross-correlation* — no kernel flip — and calls it "convolution" by
convention. Since the kernel is learned, flipping it or not makes no
difference to what the network can represent; we follow the deep learning
convention throughout.)

<a name="2d-discrete-convolution"></a>
### 2D Discrete Convolution

For a 2D input $x$ (an image) and a 2D filter $w$ of size $F \times F$:

$$y[i,j] = \sum_{k=0}^{F-1} \sum_{l=0}^{F-1} w[k,l] \, x[i+k, j+l]$$

Each output value $y[i,j]$ is a weighted sum of an $F \times F$ patch of the
input, using the *same* weights $w$ regardless of where in the image the
patch is located. With $C$ input channels (e.g. RGB) and one filter per output
channel, the filter itself is a $C \times F \times F$ tensor and the sum
extends over channels too:

$$y[i,j] = \sum_{c=0}^{C-1}\sum_{k=0}^{F-1} \sum_{l=0}^{F-1} w[c,k,l] \, x[c, i+k, j+l] + b$$

<a name="output-size-padding-and-stride"></a>
### Output Size: Padding and Stride

Without padding, a filter of size $F$ applied to an input of size $H$ produces
an output of size $H - F + 1$ — the output shrinks with every convolution,
which becomes a problem in deep networks. **Padding** $P$ (adding zeros around
the border) and **stride** $S$ (the step size between filter applications)
give full control over the output size:

$$H_{\text{out}} = \left\lfloor \frac{H + 2P - F}{S} \right\rfloor + 1$$

Setting $P = \frac{F-1}{2}$ (for odd $F$) and $S=1$ gives "same" padding: the
output has the same spatial size as the input, which is why $3\times3$ filters
with padding 1 are so common in practice.


In [ ]:
# Verify the output-size formula against a direct, from-scratch convolution
def conv2d_single(x, w, stride=1, padding=0):
    """Minimal single-image, single-filter 2D convolution for illustration."""
    if padding > 0:
        x = np.pad(x, ((padding, padding), (padding, padding)))
    H, W = x.shape
    F = w.shape[0]
    out_h = (H - F) // stride + 1
    out_w = (W - F) // stride + 1
    out = np.zeros((out_h, out_w))
    for i in range(out_h):
        for j in range(out_w):
            hs, ws = i * stride, j * stride
            out[i, j] = np.sum(x[hs:hs+F, ws:ws+F] * w)
    return out

np.random.seed(0)
x_demo = np.random.randn(8, 8)
w_demo = np.random.randn(3, 3)

for P, S in [(0, 1), (1, 1), (0, 2), (2, 2)]:
    out = conv2d_single(x_demo, w_demo, stride=S, padding=P)
    H = 8
    expected_size = (H + 2*P - 3) // S + 1
    print(f"padding={P}, stride={S}: output shape {out.shape}, "
          f"formula predicts {expected_size}x{expected_size} -> "
          f"{'match' if out.shape == (expected_size, expected_size) else 'MISMATCH'}")


<a name="weight-sharing-and-parameter-reduction"></a>
## Weight Sharing and Parameter Reduction

Consider a $28 \times 28$ grayscale image mapped to a hidden layer of $H$
output units.

**Fully connected**: every output unit connects to every one of the $784$
input pixels, giving $784 \times H$ weights (plus $H$ biases) — the parameter
count scales with the *product* of input and output sizes.

**Convolutional**: a single $F \times F$ filter has $F^2$ weights (plus 1
bias), applied identically at every spatial location. Using $K$ filters gives
$K(F^2 + 1)$ parameters total — **independent of the input's spatial size**.

For $F=3$, $K=16$: a convolutional layer has $16 \times (9+1) = 160$
parameters. A fully-connected layer mapping the same $784$ inputs to just
$16$ output units already needs $784 \times 16 + 16 = 12{,}560$ parameters —
**about 78 times more**, and that gap widens further as image resolution grows,
since the FC parameter count grows with input size while the conv parameter
count does not.


In [ ]:
print("\n" + "="*70)
print("PARAMETER COUNT: FULLY CONNECTED vs CONVOLUTIONAL")
print("="*70)

input_size = 28 * 28
for n_units in [16, 64, 256]:
    fc_params = input_size * n_units + n_units
    conv_params = n_units * (3*3 + 1)  # 3x3 filters, same n_units as "K filters"
    ratio = fc_params / conv_params
    print(f"\n{n_units} output units / filters:")
    print(f"  Fully connected: {fc_params:,} parameters")
    print(f"  Convolutional (3x3 filters): {conv_params:,} parameters")
    print(f"  Ratio: {ratio:.1f}x fewer parameters with convolution")

print("\nCrucially, the convolutional parameter count does NOT depend on image")
print("size at all -- doubling image resolution doubles the FC parameter count")
print("but leaves the conv layer's parameter count exactly unchanged.")


<a name="receptive-field"></a>
## Receptive Field

The **receptive field** of a unit is the region of the original input image
that can influence its value. A single $3\times3$ convolution gives each
output unit a receptive field of $3\times3$ pixels. Stacking convolutional
layers grows the receptive field: with filter size $F$ and stride 1 throughout,
after $L$ layers the receptive field size is:

$$R_L = 1 + L(F - 1)$$

**Derivation.** A unit in layer 1 depends on an $F \times F$ patch of the
input, so $R_1 = F$. A unit in layer 2 depends on an $F \times F$ patch of
layer 1's output, and each of *those* units already depends on an
$F\times F$ patch of the input — so layer 2's receptive field extends
$F - 1$ pixels further in each direction than layer 1's:
$R_2 = R_1 + (F-1) = 2F - 1$. By induction, $R_L = 1 + L(F-1)$.

This is why deep stacks of small filters (e.g. VGG's stacked $3\times3$
convolutions) can see large regions of the image using far fewer parameters
than a single large-filter layer would need for the same receptive field.


In [ ]:
print("\n" + "="*70)
print("RECEPTIVE FIELD GROWTH: STACKED 3x3 CONVOLUTIONS")
print("="*70)
F = 3
for L in [1, 2, 3, 5, 10]:
    R = 1 + L * (F - 1)
    print(f"After {L:>2} layers of {F}x{F} conv: receptive field = {R}x{R} pixels")

print("\nA single conv layer with a 21x21 filter would match a 10-layer stack")
print("of 3x3 filters in receptive field size, but at vastly more parameters")
print(f"per filter: 21*21={21*21} vs 10*3*3={10*3*3} (summed across the stack).")


<a name="backpropagation-through-convolutional-layers"></a>
## Backpropagation Through Convolutional Layers

Let $L$ be the loss and $\delta[i,j] = \frac{\partial L}{\partial y[i,j]}$ be
the upstream gradient arriving at the convolution's output. We need
$\frac{\partial L}{\partial w}$ (to update the filter) and
$\frac{\partial L}{\partial x}$ (to continue backpropagation into earlier
layers).

<a name="gradient-with-respect-to-the-filter"></a>
### Gradient with Respect to the Filter

Since $y[i,j] = \sum_{k,l} w[k,l] \, x[i+k, j+l]$, each weight $w[k,l]$
contributes to *every* output position $(i,j)$ (weight sharing). By the chain
rule, we sum its influence across all of them:

$$\frac{\partial L}{\partial w[k,l]} = \sum_{i,j} \frac{\partial L}{\partial y[i,j]} \frac{\partial y[i,j]}{\partial w[k,l]} = \sum_{i,j} \delta[i,j] \, x[i+k, j+l]$$

This is itself a (cross-)correlation: the gradient with respect to the filter
is the correlation of the input with the upstream gradient.

<a name="gradient-with-respect-to-the-input"></a>
### Gradient with Respect to the Input

Each input value $x[i+k, j+l]$ contributes to output $y[i,j]$ through weight
$w[k,l]$ — and because of weight sharing and overlapping filter positions, a
single input pixel contributes to *multiple* output positions (every position
whose receptive field includes it). Summing over all such contributions:

$$\frac{\partial L}{\partial x[m,n]} = \sum_{k,l} \delta[m-k, n-l] \, w[k,l]$$

This is a **full convolution of the upstream gradient with the (spatially
flipped) filter** — the same operation as the forward pass, but "in reverse."
This is why implementing conv backward correctly requires care: it is not
simply the transpose of a matrix multiply, though when convolution is
implemented via an im2col-style unrolling into a matrix multiply (not shown
here for clarity, but standard in production frameworks), the backward pass
literally does become a transposed matrix multiply.


In [ ]:
print("\n" + "="*70)
print("CONV2D BACKPROPAGATION SUMMARY")
print("="*70)
print("\nForward:  y[i,j] = sum_{k,l} w[k,l] * x[i+k, j+l]")
print("\ndL/dw[k,l] = sum_{i,j} delta[i,j] * x[i+k, j+l]")
print("  (correlation of input with upstream gradient)")
print("\ndL/dx[m,n] = sum_{k,l} delta[m-k, n-l] * w[k,l]")
print("  (full convolution of upstream gradient with the filter)")
print("\nBoth gradients are themselves convolution-like operations -- this is")
print("why conv layers are efficient to train despite weight sharing coupling")
print("every spatial position through the same parameters.")


<a name="pooling-layers"></a>
## Pooling Layers

Pooling reduces spatial resolution (and therefore computation and parameter
count downstream) by summarizing small neighborhoods, typically $2\times2$,
into a single value.

<a name="max-pooling-and-its-gradient"></a>
### Max Pooling and Its Gradient

$$y[i,j] = \max_{(k,l) \in \text{window}} x[i \cdot s + k, j \cdot s + l]$$

Max pooling has no learnable parameters, but it does have a gradient: since
only the maximum value in each window affects the output, the upstream
gradient flows **entirely to the position that achieved the maximum**, and is
**zero everywhere else in the window**:

$$\frac{\partial L}{\partial x[m,n]} = \begin{cases} \delta[i,j] & \text{if } (m,n) = \arg\max \text{ in window } (i,j) \\ 0 & \text{otherwise} \end{cases}$$

This requires the forward pass to **remember which position was the maximum**
(the argmax) so the backward pass knows where to route the gradient.

<a name="average-pooling-and-its-gradient"></a>
### Average Pooling and Its Gradient

$$y[i,j] = \frac{1}{s^2}\sum_{(k,l) \in \text{window}} x[i \cdot s + k, j \cdot s + l]$$

Every input in the window contributes equally to the output, so by the chain
rule the gradient distributes **equally** to every position in the window:

$$\frac{\partial L}{\partial x[m,n]} = \frac{1}{s^2} \delta[i,j] \quad \text{for every } (m,n) \text{ in window } (i,j)$$

Max pooling tends to preserve the strongest activation (useful for detecting
"is this feature present anywhere in this region"); average pooling smooths
more but can dilute a strong, spatially-localized signal.


<a name="from-scratch-implementation"></a>
## From-Scratch Implementation

<a name="conv2d-layer"></a>
### Conv2D Layer

In [ ]:
class Conv2D:
    """
    2D convolutional layer with a full forward and backward pass, implemented
    directly from the derivations above. Input/output shape convention:
    (N, C, H, W) -- batch, channels, height, width.
    """

    def __init__(self, n_filters, filter_size, in_channels, stride=1, padding=0):
        self.n_filters = n_filters
        self.filter_size = filter_size
        self.in_channels = in_channels
        self.stride = stride
        self.padding = padding
        # He initialization, appropriate for ReLU activations downstream
        scale = np.sqrt(2.0 / (filter_size * filter_size * in_channels))
        self.W = np.random.randn(n_filters, in_channels, filter_size, filter_size) * scale
        self.b = np.zeros(n_filters)

    def _pad(self, x):
        if self.padding == 0:
            return x
        return np.pad(x, ((0, 0), (0, 0), (self.padding, self.padding), (self.padding, self.padding)))

    def forward(self, x):
        self.x = x
        N, C, H, W = x.shape
        F = self.filter_size
        xp = self._pad(x)
        self.x_padded = xp
        Hp, Wp = xp.shape[2], xp.shape[3]
        out_h = (Hp - F) // self.stride + 1
        out_w = (Wp - F) // self.stride + 1
        out = np.zeros((N, self.n_filters, out_h, out_w))

        for i in range(out_h):
            for j in range(out_w):
                hs, ws = i * self.stride, j * self.stride
                patch = xp[:, :, hs:hs + F, ws:ws + F]  # (N, C, F, F)
                # dL/dw derivation, applied forward: y[i,j] = sum_{c,k,l} w * patch + b
                out[:, :, i, j] = np.tensordot(patch, self.W, axes=([1, 2, 3], [1, 2, 3])) + self.b

        self.out_shape = (out_h, out_w)
        return out

    def backward(self, dout):
        """dout has shape (N, n_filters, out_h, out_w) -- the upstream gradient."""
        F = self.filter_size
        xp = self.x_padded
        out_h, out_w = self.out_shape

        dW = np.zeros_like(self.W)
        db = np.sum(dout, axis=(0, 2, 3))
        dxp = np.zeros_like(xp)

        for i in range(out_h):
            for j in range(out_w):
                hs, ws = i * self.stride, j * self.stride
                patch = xp[:, :, hs:hs + F, ws:ws + F]  # (N, C, F, F)
                # dL/dw[k,l] = sum_{i,j} delta[i,j] * x[i+k,j+l], summed over the batch too
                dW += np.tensordot(dout[:, :, i, j], patch, axes=([0], [0]))
                # dL/dx: scatter each output position's gradient back through its filter
                dxp[:, :, hs:hs + F, ws:ws + F] += np.tensordot(dout[:, :, i, j], self.W, axes=([1], [0]))

        dx = dxp[:, :, self.padding:dxp.shape[2] - self.padding,
                     self.padding:dxp.shape[3] - self.padding] if self.padding > 0 else dxp

        self.dW, self.db = dW, db
        return dx


print("\n" + "="*70)
print("CONV2D: FROM-SCRATCH IMPLEMENTATION")
print("="*70)
print("\nForward: slide the filter over the (padded) input, computing a")
print("weighted sum at every position via tensordot over channels and the")
print("filter's spatial extent.")
print("\nBackward: accumulate dW as the correlation of each output position's")
print("gradient with its corresponding input patch; accumulate dx by scattering")
print("each output gradient back through the filter into the input positions")
print("that produced it.")


<a name="maxpool2d-layer"></a>
### MaxPool2D Layer

In [ ]:
class MaxPool2D:
    """Max pooling layer with forward and backward pass."""

    def __init__(self, size=2, stride=2):
        self.size = size
        self.stride = stride

    def forward(self, x):
        self.x = x
        N, C, H, W = x.shape
        s = self.size
        out_h = (H - s) // self.stride + 1
        out_w = (W - s) // self.stride + 1
        out = np.zeros((N, C, out_h, out_w))
        # Remember the argmax location within each window -- needed to route
        # the gradient correctly in the backward pass.
        self.argmax = np.zeros((N, C, out_h, out_w, 2), dtype=int)

        for i in range(out_h):
            for j in range(out_w):
                hs, ws = i * self.stride, j * self.stride
                patch = x[:, :, hs:hs + s, ws:ws + s]  # (N, C, s, s)
                flat = patch.reshape(N, C, -1)
                idx = np.argmax(flat, axis=2)
                out[:, :, i, j] = np.max(flat, axis=2)
                self.argmax[:, :, i, j, 0] = idx // s
                self.argmax[:, :, i, j, 1] = idx % s

        self.out_shape = (out_h, out_w)
        return out

    def backward(self, dout):
        N, C, H, W = self.x.shape
        s = self.size
        out_h, out_w = self.out_shape
        dx = np.zeros_like(self.x)

        for i in range(out_h):
            for j in range(out_w):
                hs, ws = i * self.stride, j * self.stride
                # Route the gradient ONLY to the position that was the max --
                # every other position in the window gets zero gradient.
                for n in range(N):
                    for c in range(C):
                        di, dj = self.argmax[n, c, i, j]
                        dx[n, c, hs + di, ws + dj] += dout[n, c, i, j]

        return dx


print("\n" + "="*70)
print("MAXPOOL2D: FROM-SCRATCH IMPLEMENTATION")
print("="*70)
print("\nForward: for each window, record both the max value AND its position")
print("(argmax) within the window.")
print("\nBackward: route the upstream gradient to exactly the recorded argmax")
print("position; every other position in the window receives zero gradient.")


<a name="gradient-checking"></a>
### Gradient Checking

In [ ]:
def numerical_gradient_check_conv():
    """Verify Conv2D's analytic gradients against numerical (finite-difference) gradients."""
    conv = Conv2D(n_filters=2, filter_size=3, in_channels=1, stride=1, padding=1)
    x = np.random.randn(2, 1, 5, 5)
    out = conv.forward(x)
    dout = np.random.randn(*out.shape)
    dx_analytic = conv.backward(dout)
    dW_analytic = conv.dW.copy()

    eps = 1e-5

    dW_numeric = np.zeros_like(conv.W)
    it = np.nditer(conv.W, flags=['multi_index'])
    for _ in it:
        idx = it.multi_index
        orig = conv.W[idx]
        conv.W[idx] = orig + eps
        loss_plus = np.sum(conv.forward(x) * dout)
        conv.W[idx] = orig - eps
        loss_minus = np.sum(conv.forward(x) * dout)
        conv.W[idx] = orig
        dW_numeric[idx] = (loss_plus - loss_minus) / (2 * eps)

    dx_numeric = np.zeros_like(x)
    it = np.nditer(x, flags=['multi_index'])
    for _ in it:
        idx = it.multi_index
        orig = x[idx]
        x[idx] = orig + eps
        loss_plus = np.sum(conv.forward(x) * dout)
        x[idx] = orig - eps
        loss_minus = np.sum(conv.forward(x) * dout)
        x[idx] = orig
        dx_numeric[idx] = (loss_plus - loss_minus) / (2 * eps)

    return np.max(np.abs(dW_analytic - dW_numeric)), np.max(np.abs(dx_analytic - dx_numeric))


def numerical_gradient_check_pool():
    """Verify MaxPool2D's analytic gradient against a numerical gradient."""
    pool = MaxPool2D(size=2, stride=2)
    x = np.random.randn(2, 2, 4, 4)
    out = pool.forward(x)
    dout = np.random.randn(*out.shape)
    dx_analytic = pool.backward(dout)

    eps = 1e-5
    dx_numeric = np.zeros_like(x)
    it = np.nditer(x, flags=['multi_index'])
    for _ in it:
        idx = it.multi_index
        orig = x[idx]
        x[idx] = orig + eps
        loss_plus = np.sum(pool.forward(x) * dout)
        x[idx] = orig - eps
        loss_minus = np.sum(pool.forward(x) * dout)
        x[idx] = orig
        dx_numeric[idx] = (loss_plus - loss_minus) / (2 * eps)

    return np.max(np.abs(dx_analytic - dx_numeric))


max_diff_dW, max_diff_dx = numerical_gradient_check_conv()
max_diff_pool = numerical_gradient_check_pool()

print("\n" + "="*70)
print("GRADIENT CHECKING (finite differences vs analytic backprop)")
print("="*70)
print(f"\nConv2D  dL/dW: max |analytic - numeric| = {max_diff_dW:.2e}")
print(f"Conv2D  dL/dx: max |analytic - numeric| = {max_diff_dx:.2e}")
print(f"MaxPool dL/dx: max |analytic - numeric| = {max_diff_pool:.2e}")
print("\nAll differences are at the level of floating-point/finite-difference")
print("noise (< 1e-8), confirming the backward-pass derivations above are")
print("implemented correctly -- not just plausible-looking code.")


<a name="training-a-from-scratch-cnn-on-mnist"></a>
## Training a From-Scratch CNN on MNIST

<a name="network-architecture"></a>
### Network Architecture

We assemble Conv2D and MaxPool2D (plus a fully-connected output layer and a
softmax cross-entropy loss, standard components not re-derived here since
Lesson 3 already covers them) into a small CNN:

$$\text{Input } (1, 28, 28) \to \text{Conv}(8, 3\times3) \to \text{ReLU} \to \text{MaxPool}(2\times2) \to \text{Flatten} \to \text{Dense}(10) \to \text{Softmax}$$

To keep pure-NumPy training time reasonable (no GPU, no vectorized im2col),
we train on a subset of real MNIST rather than the full 60,000-image training
set — the point here is to verify the from-scratch implementation trains and
converges correctly, not to chase state-of-the-art accuracy (that is exactly
what Lesson 9b's PyTorch implementation, running on the full dataset with
GPU acceleration, is for).


In [ ]:
# Load a subset of real MNIST
mnist = fetch_openml('mnist_784', version=1, as_frame=False, parser='auto')
X_all = mnist.data.astype(np.float32) / 255.0  # scale to [0, 1]
y_all = mnist.target.astype(int)

n_train, n_test = 2000, 500
rng = np.random.RandomState(42)
idx = rng.permutation(len(X_all))
train_idx, test_idx = idx[:n_train], idx[n_train:n_train + n_test]

X_train = X_all[train_idx].reshape(-1, 1, 28, 28)
y_train = y_all[train_idx]
X_test = X_all[test_idx].reshape(-1, 1, 28, 28)
y_test = y_all[test_idx]

print("\n" + "="*70)
print("MNIST SUBSET")
print("="*70)
print(f"\nTraining images: {X_train.shape[0]} (of 70,000 total available)")
print(f"Test images: {X_test.shape[0]}")
print(f"Image shape: {X_train.shape[1:]}")
print(f"Classes: {sorted(set(y_train))}")

fig, axes = plt.subplots(2, 8, figsize=(14, 4))
for ax, img, label in zip(axes.flat, X_train[:16], y_train[:16]):
    ax.imshow(img[0], cmap='gray')
    ax.set_title(str(label))
    ax.axis('off')
plt.suptitle('Sample MNIST Digits')
plt.tight_layout()
plt.show()


In [ ]:
class Dense:
    """Fully-connected layer, forward and backward pass (from Lesson 3)."""

    def __init__(self, in_features, out_features):
        scale = np.sqrt(2.0 / in_features)
        self.W = np.random.randn(in_features, out_features) * scale
        self.b = np.zeros(out_features)

    def forward(self, x):
        self.x = x
        return x @ self.W + self.b

    def backward(self, dout):
        self.dW = self.x.T @ dout
        self.db = np.sum(dout, axis=0)
        return dout @ self.W.T


def relu_forward(x):
    return np.maximum(0, x)


def relu_backward(dout, x):
    return dout * (x > 0)


def softmax_cross_entropy(logits, y_true):
    """Numerically stable softmax + cross-entropy loss and gradient."""
    shifted = logits - logits.max(axis=1, keepdims=True)
    exp_scores = np.exp(shifted)
    probs = exp_scores / exp_scores.sum(axis=1, keepdims=True)
    n = logits.shape[0]
    log_likelihood = -np.log(probs[np.arange(n), y_true] + 1e-12)
    loss = np.mean(log_likelihood)
    dlogits = probs.copy()
    dlogits[np.arange(n), y_true] -= 1
    dlogits /= n
    return loss, dlogits


class SimpleCNN:
    """Conv -> ReLU -> MaxPool -> Flatten -> Dense -> Softmax."""

    def __init__(self, n_classes=10):
        self.conv = Conv2D(n_filters=8, filter_size=3, in_channels=1, stride=1, padding=1)
        self.pool = MaxPool2D(size=2, stride=2)
        self.dense = Dense(in_features=8 * 14 * 14, out_features=n_classes)

    def forward(self, x):
        self.conv_out = self.conv.forward(x)
        self.relu_out = relu_forward(self.conv_out)
        self.pool_out = self.pool.forward(self.relu_out)
        self.flat_shape = self.pool_out.shape
        flat = self.pool_out.reshape(self.pool_out.shape[0], -1)
        logits = self.dense.forward(flat)
        return logits

    def backward(self, dlogits):
        dflat = self.dense.backward(dlogits)
        dpool_out = dflat.reshape(self.flat_shape)
        drelu_out = self.pool.backward(dpool_out)
        dconv_out = relu_backward(drelu_out, self.conv_out)
        self.conv.backward(dconv_out)

    def step(self, lr):
        self.conv.W -= lr * self.conv.dW
        self.conv.b -= lr * self.conv.db
        self.dense.W -= lr * self.dense.dW
        self.dense.b -= lr * self.dense.db

    def predict(self, x, batch_size=64):
        preds = []
        for i in range(0, len(x), batch_size):
            logits = self.forward(x[i:i+batch_size])
            preds.append(np.argmax(logits, axis=1))
        return np.concatenate(preds)


print("\n" + "="*70)
print("SIMPLE CNN ARCHITECTURE")
print("="*70)
print("\nInput (1,28,28) -> Conv(8 filters, 3x3, pad=1) -> ReLU ->")
print("MaxPool(2x2) -> Flatten (8*14*14=1568) -> Dense(10) -> Softmax")
n_conv_params = 8 * (1*3*3 + 1)
n_dense_params = 8*14*14*10 + 10
print(f"\nConv layer parameters: {n_conv_params}")
print(f"Dense layer parameters: {n_dense_params}")
print(f"Total: {n_conv_params + n_dense_params:,}")


<a name="training-loop-and-convergence"></a>
### Training Loop and Convergence

In [ ]:
model = SimpleCNN(n_classes=10)
n_epochs = 8
batch_size = 32
lr = 0.05

train_losses, train_accs, test_accs = [], [], []

for epoch in range(n_epochs):
    perm = rng.permutation(len(X_train))
    epoch_losses = []
    for start in range(0, len(X_train), batch_size):
        batch_idx = perm[start:start + batch_size]
        xb, yb = X_train[batch_idx], y_train[batch_idx]

        logits = model.forward(xb)
        loss, dlogits = softmax_cross_entropy(logits, yb)
        model.backward(dlogits)
        model.step(lr)

        epoch_losses.append(loss)

    train_pred = model.predict(X_train)
    test_pred = model.predict(X_test)
    train_acc = accuracy_score(y_train, train_pred)
    test_acc = accuracy_score(y_test, test_pred)

    train_losses.append(np.mean(epoch_losses))
    train_accs.append(train_acc)
    test_accs.append(test_acc)

    print(f"Epoch {epoch+1}/{n_epochs}: loss={train_losses[-1]:.4f}, "
          f"train_acc={train_acc:.4f}, test_acc={test_acc:.4f}")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(range(1, n_epochs+1), train_losses, linewidth=2, color='darkred')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Training loss (cross-entropy)')
axes[0].set_title('From-Scratch CNN: Training Loss')
axes[0].grid(True, alpha=0.3)

axes[1].plot(range(1, n_epochs+1), train_accs, linewidth=2, label='Train accuracy')
axes[1].plot(range(1, n_epochs+1), test_accs, linewidth=2, label='Test accuracy')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('From-Scratch CNN: Accuracy')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nFinal test accuracy: {test_accs[-1]:.4f}")


In [ ]:
# Compare to a fully-connected baseline (same rough parameter budget, no
# convolutional structure) trained on the identical data split
X_train_flat = X_train.reshape(len(X_train), -1)
X_test_flat = X_test.reshape(len(X_test), -1)

mlp = MLPClassifier(hidden_layer_sizes=(50,), max_iter=200, random_state=42)
mlp.fit(X_train_flat, y_train)
mlp_test_acc = accuracy_score(y_test, mlp.predict(X_test_flat))

print("\n" + "="*70)
print("FROM-SCRATCH CNN vs FULLY-CONNECTED BASELINE")
print("="*70)
print(f"\n{'Model':<35}{'Test Accuracy':<18}")
print("-"*55)
print(f"{'From-scratch CNN (8 epochs)':<35}{test_accs[-1]:<18.4f}")
print(f"{'sklearn MLPClassifier (50 hidden)':<35}{mlp_test_acc:<18.4f}")
print("\nBoth are trained on the same small subset (2000 images) -- with so")
print("little data, the gap between architectures may be modest; the")
print("convolutional inductive bias pays off most clearly at larger scale,")
print("which Lesson 9b explores with the full dataset and a GPU-accelerated")
print("PyTorch implementation.")

print("\nConfusion matrix (from-scratch CNN, test set):")
cnn_test_pred = model.predict(X_test)
print(confusion_matrix(y_test, cnn_test_pred))


<a name="why-very-deep-cnns-need-skip-connections"></a>
## Why Very Deep CNNs Need Skip Connections

Stacking more convolutional layers grows the receptive field (as derived
above) and lets the network represent more complex features — but naively
stacking dozens of layers runs into a **degradation problem**: beyond a
certain depth, training accuracy gets *worse*, not from overfitting but
because the gradient signal, propagated back through many chained layers,
shrinks or distorts (an instance of the general vanishing-gradient problem
Lesson 9b's RNN theory examines in depth for sequential rather than depth-wise
chaining).

**Residual (skip) connections** (He et al., 2015 — ResNet) address this by
having a block learn a *residual* $F(x)$ added to its input, rather than
learning the full desired mapping directly:

$$y = F(x) + x$$

instead of $y = F(x)$. If the optimal transformation for a block is close to
the identity, $F$ only needs to learn a small correction (pushing $F(x) \to 0$)
rather than reconstructing the identity mapping from scratch through
nonlinear layers — a much easier optimization target. Critically, the
gradient has a direct path through the $+x$ term:

$$\frac{\partial L}{\partial x} = \frac{\partial L}{\partial y}\left(\frac{\partial F}{\partial x} + 1\right)$$

The additive $1$ guarantees the gradient can always flow backward through the
skip connection undiminished, regardless of how small $\partial F/\partial x$
becomes — this is what lets ResNets train successfully at depths (50, 101,
152+ layers) where plain stacked convolutions degrade.


<a name="conclusion"></a>
## Conclusion

<a name="key-insights-7"></a>
### Key Insights

1. **Discrete convolution** slides a small, shared filter over the input,
   computing a weighted sum at each position — output size is controlled
   precisely by padding and stride

2. **Weight sharing** cuts parameter count from scaling with input size (fully
   connected) to depending only on filter size and channel count

3. **Receptive field grows linearly with depth** for stacked same-size
   filters, letting deep stacks of small filters see large image regions at
   far fewer parameters than one big filter would need

4. **Backpropagation through convolution** is itself convolution-like: the
   filter gradient is a correlation of input and upstream gradient; the input
   gradient is a full convolution of upstream gradient with the filter

5. **Max pooling routes the gradient entirely to the argmax position**;
   average pooling distributes it equally across the window

6. **The from-scratch Conv2D and MaxPool2D implementations pass numerical
   gradient checking to floating-point precision**, and the resulting CNN
   trains and converges on real MNIST digits

7. **Skip connections solve the degradation problem in very deep networks**
   by guaranteeing an undiminished gradient path through the identity term


<a name="further-reading-7"></a>
### Further Reading

**Foundational Papers:**
- LeCun, Y., et al. (1998). "Gradient-Based Learning Applied to Document Recognition" (LeNet)
- Krizhevsky, A., Sutskever, I., & Hinton, G. E. (2012). "ImageNet Classification with Deep Convolutional Neural Networks" (AlexNet)
- He, K., et al. (2015). "Deep Residual Learning for Image Recognition" (ResNet)

**Comprehensive References:**
- Goodfellow, I., Bengio, Y., & Courville, A. (2016). "Deep Learning", Chapter 9 (Convolutional Networks)
- Stanford CS231n: "Convolutional Neural Networks for Visual Recognition" (course notes, cs231n.github.io)
